# Tugas Praktikum 5 — Sinkronisasi Thread

**Nama:** Najwa Azzahra  
**NIM:** 2411513004  

Notebook ini berisi implementasi dan pengamatan empat latihan: `RLock`, deadlock tiga lock, bounded buffer, serta refleksi singkat.

> Jalankan setiap sel kode secara berurutan. Hasil keluaran dapat sedikit berbeda karena penjadwalan thread oleh sistem operasi.

## 1. RLock pada Class `RekeningBank`

Class `RekeningBank` memiliki method `setor()`, `tarik()`, dan `transfer()`. Setiap objek rekening mempunyai `RLock` sendiri. Method `transfer()` memanggil `tarik()` pada rekening asal dan `setor()` pada rekening tujuan.

`RLock` diperlukan karena lock yang sama dapat diambil kembali oleh thread yang sudah memilikinya. Pada proses transfer, lock rekening asal sudah diambil oleh `transfer()`, kemudian `tarik()` mengambil lock rekening asal sekali lagi. Jika menggunakan `Lock` biasa, thread akan menunggu lock yang sedang dipegangnya sendiri sehingga terjadi deadlock. Dua lock rekening juga diambil berdasarkan urutan identitas objek agar dua transfer berlawanan arah tidak saling menunggu.

In [ ]:
from threading import RLock

class RekeningBank:
    def __init__(self, nama, saldo=0):
        self.nama = nama
        self.saldo = saldo
        self._lock = RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setoran harus lebih dari 0.")
        with self._lock:
            self.saldo += jumlah
            return self.saldo

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah penarikan harus lebih dari 0.")
        with self._lock:
            if self.saldo < jumlah:
                raise ValueError("Saldo tidak mencukupi.")
            self.saldo -= jumlah
            return self.saldo

    def transfer(self, rekening_tujuan, jumlah):
        if rekening_tujuan is self:
            raise ValueError("Rekening asal dan tujuan tidak boleh sama.")
        if jumlah <= 0:
            raise ValueError("Jumlah transfer harus lebih dari 0.")

        # Ambil kedua lock dalam urutan konsisten untuk mencegah deadlock.
        pertama, kedua = sorted(
            [self, rekening_tujuan], key=id
        )
        with pertama._lock:
            with kedua._lock:
                self.tarik(jumlah)
                rekening_tujuan.setor(jumlah)

        return f"Transfer Rp{jumlah:,} dari {self.nama} ke {rekening_tujuan.nama} berhasil."

### Pengujian

Saldo awal rekening A adalah Rp1.000.000 dan rekening B Rp500.000. Setelah transfer Rp200.000, saldo A berkurang dan saldo B bertambah dengan jumlah yang sama. Total saldo kedua rekening tetap.

In [ ]:
rekening_a = RekeningBank("A", 1_000_000)
rekening_b = RekeningBank("B", 500_000)

print("Saldo sebelum transfer:")
print(rekening_a.nama, rekening_a.saldo)
print(rekening_b.nama, rekening_b.saldo)

print("\\n" + rekening_a.transfer(rekening_b, 200_000))

print("\\nSaldo setelah transfer:")
print(rekening_a.nama, rekening_a.saldo)
print(rekening_b.nama, rekening_b.saldo)
print("Total saldo:", rekening_a.saldo + rekening_b.saldo)

**Analisis:** `RLock` membuat pemanggilan method yang saling memanggil tetap aman karena thread yang sama boleh mengambil lock berulang kali. Penguncian kedua rekening selama transfer menjaga agar perubahan saldo berlangsung sebagai satu operasi yang konsisten. Urutan pengambilan lock yang tetap juga mencegah dua transfer berlawanan arah membentuk circular wait.

## 2. Deadlock dengan Tiga Lock dan Tiga Thread

Skenario dibuat dengan tiga lock (A, B, C) dan tiga thread. Setiap thread mengambil satu lock terlebih dahulu, lalu mencoba mengambil lock berikutnya. Polanya membentuk circular wait: Thread 1 menunggu B, Thread 2 menunggu C, dan Thread 3 menunggu A.

Eksperimen dijalankan di proses terpisah menggunakan `subprocess.run()` dan `timeout`. Dengan begitu, proses yang mengalami deadlock dapat dihentikan otomatis dan notebook utama tidak ikut macet.

In [ ]:
import subprocess
import sys
import textwrap

program_deadlock = r'''
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()
barrier = threading.Barrier(3)

def thread_1():
    with lock_a:
        print("Thread 1 memegang Lock-A", flush=True)
        barrier.wait()
        print("Thread 1 menunggu Lock-B", flush=True)
        lock_b.acquire()

def thread_2():
    with lock_b:
        print("Thread 2 memegang Lock-B", flush=True)
        barrier.wait()
        print("Thread 2 menunggu Lock-C", flush=True)
        lock_c.acquire()

def thread_3():
    with lock_c:
        print("Thread 3 memegang Lock-C", flush=True)
        barrier.wait()
        print("Thread 3 menunggu Lock-A", flush=True)
        lock_a.acquire()

threads = [
    threading.Thread(target=thread_1),
    threading.Thread(target=thread_2),
    threading.Thread(target=thread_3)
]

for thread in threads:
    thread.start()

# Join tanpa timeout membuat proses tetap hidup jika deadlock terjadi.
for thread in threads:
    thread.join()
'''

try:
    hasil = subprocess.run(
        [sys.executable, "-u", "-c", program_deadlock],
        capture_output=True,
        text=True,
        timeout=3
    )
    print(hasil.stdout)
    print("Proses selesai tanpa timeout.")
except subprocess.TimeoutExpired as error:
    keluaran = error.stdout or ""
    if isinstance(keluaran, bytes):
        keluaran = keluaran.decode("utf-8", errors="replace")
    print(keluaran)
    print("TIMEOUT: proses dihentikan karena ketiga thread saling menunggu.")

**Analisis:** Deadlock terjadi karena terdapat circular wait. Thread 1 memegang A dan menunggu B, Thread 2 memegang B dan menunggu C, sedangkan Thread 3 memegang C dan menunggu A. Tidak ada thread yang dapat melanjutkan pekerjaannya untuk melepaskan lock. `timeout` membatasi waktu eksperimen sehingga proses yang macet tidak membuat notebook berhenti merespons.

## 3. Bounded Buffer dengan Kapasitas Berbeda

Eksperimen menggunakan dua producer dan dua consumer dengan kapasitas buffer 1, 3, dan 10. Setiap producer menghasilkan lima data. `queue.Queue` digunakan karena sudah menyediakan mekanisme sinkronisasi untuk operasi `put()` dan `get()`.

Kapasitas buffer dapat memengaruhi seberapa sering producer harus menunggu karena buffer penuh dan consumer harus menunggu karena buffer kosong. Urutan interleaving dapat berubah pada setiap eksekusi karena bergantung pada penjadwalan thread.

In [ ]:
import queue
import threading
import time

def eksperimen_buffer(kapasitas):
    buffer = queue.Queue(maxsize=kapasitas)
    jumlah_producer = 2
    jumlah_consumer = 2
    data_per_producer = 5
    sentinel = None

    log = []
    hasil_konsumen = []
    log_lock = threading.Lock()
    maksimum_terisi = [0]

    def producer(nomor):
        for i in range(1, data_per_producer + 1):
            item = f"P{nomor}-{i}"
            buffer.put(item)  # Menunggu jika buffer penuh.
            with log_lock:
                log.append(f"Producer {nomor} memasukkan {item}")
                maksimum_terisi[0] = max(
                    maksimum_terisi[0], buffer.qsize()
                )
            time.sleep(0.002)

    def consumer(nomor):
        while True:
            item = buffer.get()  # Menunggu jika buffer kosong.
            if item is sentinel:
                buffer.task_done()
                break
            with log_lock:
                log.append(f"Consumer {nomor} mengambil {item}")
                hasil_konsumen.append((nomor, item))
            buffer.task_done()
            time.sleep(0.003)

    producers = [
        threading.Thread(target=producer, args=(i,))
        for i in range(1, jumlah_producer + 1)
    ]
    consumers = [
        threading.Thread(target=consumer, args=(i,))
        for i in range(1, jumlah_consumer + 1)
    ]

    for thread in consumers:
        thread.start()
    for thread in producers:
        thread.start()

    for thread in producers:
        thread.join()

    # Kirim satu penanda berhenti untuk setiap consumer.
    for _ in range(jumlah_consumer):
        buffer.put(sentinel)

    for thread in consumers:
        thread.join()

    print(f"\\nKAPASITAS BUFFER = {kapasitas}")
    for kejadian in log:
        print(kejadian)
    print("Jumlah data diproses:", len(hasil_konsumen))
    print("Maksimum isi buffer teramati:", maksimum_terisi[0])

for ukuran in [1, 3, 10]:
    eksperimen_buffer(ukuran)

**Analisis:** Pada kapasitas 1, producer dan consumer cenderung bergantian lebih sering karena hanya satu data yang dapat berada di buffer. Kapasitas 3 memberi ruang lebih besar sehingga beberapa data dapat menunggu untuk diambil. Kapasitas 10 dapat menampung seluruh data produksi dalam eksperimen ini, sehingga producer berpeluang menyelesaikan pekerjaannya lebih cepat sebelum consumer mengambil semua data. Meskipun demikian, urutan keluaran tidak selalu sama karena proses dijalankan secara konkuren.

## 4. Refleksi Singkat

Menurut saya, deadlock paling berbahaya untuk aplikasi produksi seperti perbankan atau e-commerce karena proses dapat berhenti ketika thread saling menunggu. Akibatnya, transaksi atau layanan pengguna bisa tertahan dan sistem menjadi tidak responsif. Sementara itu, starvation juga berbahaya karena ada thread yang terus-menerus tidak memperoleh giliran untuk menggunakan resource. Livelock membuat thread tetap aktif, tetapi tidak menghasilkan kemajuan yang berarti. Masalah yang paling sulit dideteksi saat testing adalah race condition yang dapat memicu starvation atau livelock dalam kondisi tertentu, karena kemunculannya bergantung pada waktu eksekusi dan penjadwalan thread. Pengujian biasa belum tentu mengulang kondisi yang sama. Oleh karena itu, aplikasi perlu diuji dengan banyak thread, beban yang bervariasi, serta pemantauan waktu tunggu dan penggunaan resource.

---
**Kesimpulan:** Penggunaan lock perlu dirancang dengan benar agar data tetap konsisten dan thread tidak saling menghambat. `RLock` membantu saat method dalam objek yang sama saling memanggil, urutan penguncian dapat mencegah circular wait, dan kapasitas buffer memengaruhi pola kerja producer-consumer.